# Open API를 활용한 네이버 뉴스 검색

## 1. 애플리케이션 등록
https://developers.naver.com/apps/#/register

## 2. 환경변수 관리
- 등록된 애플리케이션 페이지에서 제공되는 Client ID, Secret은 절대 외부로 노출되면 안된다.
- dotenv(.env)를 통해서 관리
    - pip install dotenv 패키지 설치
    - .gitignore 파일에 .env를 무시하는 구문 추가

In [2]:
from urllib import request
!pip install python-dotenv

### 프로젝트 하위 폴더에 .env 파일을 만들어 아래 값을 입력
```
NAVER_CLIENT_ID={{Client Id}}
NAVER_CLIENT_SECRET={{Client Secret}}
```
- 네이버 개발자센터에 등록된 애플리케이션에서 확인 가능

In [3]:
# .env 파일을 로드해서 환경변수로 등록
from dotenv import load_dotenv
load_dotenv()   # .env 파일을 읽어와 자동으로 환경변수로 등록
# 읽어오기 성공 시 True 반환, 실패 시 False 반환

True

In [7]:
# 환경변수에서 .env 등록한 내용 얻어오기
import os
NAVER_CLIENT_ID=os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET=os.getenv('NAVER_CLIENT_SECRET')

# api key, 비밀번호를 print하는 구문을 실수로 남겨놓으면 노출될 가능성이 있음
# -> jupyter 변수 탭에서 확인

# .env 내용이 환경변수로 등록되지 않은 경우
if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError('네이버 클라이언트 아이디 또는 시크릿이 환경변수에 등록되지 않았습니다.')

## 3. API 요청
- 파이썬에서 웹 요청(http)을 처리하기 위해서는 `requests`라는 라이브러리가 필요

`!pip install requests`

In [8]:
!pip install requests

In [13]:
# 네이버 뉴스 검색 API 요청
import urllib.request
import socket

encText = urllib.parse.quote('인공지능')
# -> url encoding 작업
# 한글 -> %시작하는 문자로 변경

url=f'https://openapi.naver.com/v1/search/news.json?query={encText}&display=10&sort=date'
# 쿼리스트링 : 전달할 값(파라미터)기 작성된 문자열

# 요청 객체 생성
request = urllib.request.Request(url)

# API 인증 정보를 요청 헤더에 추가
request.add_header("X-Naver-Client-Id", NAVER_CLIENT_ID)
request.add_header("X-Naver-Client-Secret", NAVER_CLIENT_SECRET)

try:
    with urllib.request.urlopen(request, timeout=10) as response:
    # 지정된 주소로 요펑 -> 결과를 response로 전달 받음
    # 단, 요청 대기 시간이 10초를 초과하면 중지

        # HTTP 응답 상태 코드
        # 200이면 정상 응답 = 성공
        response_code = response.getcode()

        # 응답 본문 확인 (bytes -> UTF-8로 변환)
        response_body = response.read().decode('utf-8')

        print("response_code : ", response_code)
        print(response_body)
        # 응답 본문이 JSON(str 타입) 형태
        # -> 이용이 필요할 경우 parsing 작업 필수

except socket.timeout:
    print('요청 시간 10초 초과')

response_code :  200
{
	"lastBuildDate":"Mon, 15 Jun 2026 11:47:39 +0900",
	"total":4045761,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"대웅제약, 아크와 손잡고 아파트형 AI 헬스케어 서비스 확대",
			"originallink":"http:\/\/www.dailysmart.co.kr\/news\/articleView.html?idxno=125501",
			"link":"http:\/\/www.dailysmart.co.kr\/news\/articleView.html?idxno=125501",
			"description":"대웅제약이 <b>인공지능<\/b>(AI) 기반 디지털 헬스케어 사업 영역을 주거 공간으로 확대한다. 대웅제약은 AI 기반 만성질환 합병증 조기 스크리닝 전문기업 아크와 주거단지 기반 프리미엄 AI 헬스케어 라운지 '상벨(SANVEL)'의... ",
			"pubDate":"Mon, 15 Jun 2026 11:46:00 +0900"
		},
		{
			"title":"'제4회 스마트축산 AI 경진대회' 공모 시작",
			"originallink":"http:\/\/www.amnews.co.kr\/news\/articleView.html?idxno=73043",
			"link":"http:\/\/www.amnews.co.kr\/news\/articleView.html?idxno=73043",
			"description":"이번 대회는 <b>인공지능<\/b>과 빅데이터 기술을 활용한 축산 현장 문제해결을 주제로 개최된다. 공모 분야는 △상용화 부문과 △알고리즘 부문(기술개발 과제, 지정 과제)'으로 나뉜다. 부문별 참가 대상은 다음과 같다. 상용화... ",
			"pubDate":"Mon, 15 Jun 2026 11:46:00 +0900"
		},
		{
			"title":"&quot;인문학에 AI·기후행동

In [21]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint   #출력 시 공백 문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET,
}
params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method : 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params,  # dict -> 쿼리 스트링으로 변환(+ url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번 대인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code    #상태코드
    data = response.json()                  #응답 데이터(json -> dict 변환)

    print('response_code : ', response_code)
    #pprint(data)
    pprint(data['items'][0])    #첫번째 기사


except requests.exceptions.Timeout:
    print('요청 시간 10초 초과')
except ValueError:
    print('응답 데이터가 JSON 형식이 아닙니다')

response_code :  200
{'description': '중소 커뮤니티 운영자들은 <b>인공지능</b>(AI) 기반 필터링 도입에 큰 비용이 드는 데다 클라우드 방식의 '
                '경우에도 응용프로그래밍 인터페이스(API) 이용료 부담이 크다며 현실적 어려움을 토로하고 있다. 공영방송 '
                '거버넌스... ',
 'link': 'https://www.newsworks.co.kr/news/articleView.html?idxno=843881',
 'originallink': 'https://www.newsworks.co.kr/news/articleView.html?idxno=843881',
 'pubDate': 'Mon, 15 Jun 2026 12:02:00 +0900',
 'title': '김종철 방미통위원장 &quot;불법촬영물 유통 수익 창출 용인 못 해&quot;'}


In [1]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint   #출력 시 공백 문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET,
}

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method : 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params,  # dict -> 쿼리 스트링으로 변환(+ url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번 대인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code    #상태코드
    data = response.content                 #응답 데이터를 문자열로(xml)

    # XML 해석 파이썬 기본 내장 모듈
    # : xml.etree.ElementTree

    # XML 파일로 저장
    with open('response.xml', 'wb') as f:
        f.write(data)

    print("response.xml 저장 완료")

except requests.exceptions.Timeout:
    print('요청 시간 10초 초과')


NameError: name 'NAVER_CLIENT_ID' is not defined